# 22 homa lossy

Question: does the engine of the crate `homa` recover from loss, duplication, damage and the loss
of its link, and report `RPC_UNKNOWN` as a request asks.

The side under test is `aosabi-devtool homa-sim` over the lossy link of `homa::sim`, which drops,
duplicates and damages packets with seeded faults, loses every GRANT of the messages it picks and
goes down at a time a scenario names. The reference side is this notebook, which knows every byte
each RPC was asked to carry and checks the SHA-256 of what each end read against it, and the outcome
of every RPC against the rules of the Homa design note, first version, section 2.4, with the report
of `RPC_UNKNOWN` that `docs/graphtp.md` section 2.7 asks of Homa.

The notebook runs one half per entry of the matrix, which `half` names. The engine half is stage
H0's, over the lossy link of `homa::sim`. The socket half is stage H2's: the engine of the crate
behind `aosabi-devtool homa-unix serve` on a real Unix Socket, driven by the Python engine
`tests/harness/homa/client.py`, whose hooks drop and duplicate packets, a server engine rebuilt
over its open socket while a request is in flight, and a client that falls silent. A half an entry
does not name prints not run below, which is never a pass, since it runs as the matrix entry of its
own.

The exit number of the engine half, stage H0 of the plan's section The notebooks: every RPC
completes or fails with a named failure within its bound, 0 duplicate deliveries, 100 of 100 lost
GRANT cases recovered through RESEND, 100 of 100 lost requests restarted and completed under
`report_unknown` false and 100 of 100 failed `Failure::RpcUnknown` with 0 restarts under true,
counted in `Stats::unknown_reported`, and 0 restarts of an RPC that received a byte of its answer
under either value.

The exit number of the socket half, stage H2: 0 duplicate deliveries and every RPC completing or
failing with a named failure at 1 and 5 percent loss, 1 of 1 request in flight across the server
rebuild restarted and answered once under `report_unknown` false and failed `Failure::RpcUnknown`
with 0 restarts under true, and 1 of 1 silent client declared dead with its server RPCs dropped and
`Event::PeerGone` reported.

Judged stages: H0 and H2 of the Homa track.

In [ ]:
# Papermill overrides this cell.
# The tree under comparison.
aosabi_root = "."

# The half this entry runs: engine, the lossy link of stage H0, or socket,
# the real Unix Sockets of stage H2.
half = "engine"

# The seed of every drawn size and of the faults of every run.
seed = 20260927

# The seeds of the loss sweep, each run at every drop rate below.
seeds = 100

# The drop rates of the sweep, per million packets, comma separated: 0, 0.1,
# 1, 5 and 20 percent.
drop_rates = "0,1000,10000,50000,200000"

# The RPCs of one run of the sweep.
rpcs_per_run = 6

# The lost requests of the exit number, under each value of report_unknown.
lost_requests = 100

# The socket half: the loss rates the hooks of client.py draw, per million
# packets, the duplication beside them, the RPCs of each run, and the RPCs
# of the pty.
socket_rates = "10000,50000"
socket_duplicate_ppm = 10000
socket_rpcs = 300
pty_rpcs = 100

# The per command timeout.
timeout_s = 600

# The pinned toolchain the crate is built with.
rust_version = "1.95.0"

## Setup

The cell prints the toolchain, the installed targets and the gates of the plan's section The
commands that the notebook rests on: the build of the devtool, the lints of the workspace and the
format of the crates `homa` and `aosabi-devtool`, every cargo call with `--locked`.

In [ ]:
import hashlib
import json
import os
import random
import sys
from pathlib import Path

for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "tests" / "harness" / "__init__.py").is_file():
        sys.path.insert(0, str(_base / "tests"))
        break

from harness import paths, run

P = paths(aosabi_root)
OUT = P.out / "22_homa_lossy"
OUT.mkdir(parents=True, exist_ok=True)
DEVTOOL = P.target / "debug" / "aosabi-devtool"
CARGO_BIN = Path(os.environ.get("CARGO_HOME") or Path.home() / ".cargo") / "bin"
CARGO = CARGO_BIN / "cargo"
RUSTUP = CARGO_BIN / "rustup"
RUSTFMT = CARGO_BIN / "rustfmt"

# The gates of the plan's section The commands that this notebook rests on,
# each cargo call with --locked: the build of the seam it drives, the lints of
# the workspace, and the format of the crate homa under its own edition 2021
# and of the devtool, which rustfmt checks because cargo fmt takes no
# --locked.
GATES = [
    ("cargo build --locked -p aosabi-devtool",
     [CARGO, "build", "--locked", "-p", "aosabi-devtool"]),
    ("cargo clippy --locked --workspace --all-targets -- -D warnings",
     [CARGO, "clippy", "--locked", "--workspace", "--all-targets", "--", "-D", "warnings"]),
    ("rustfmt --check --edition 2021 crates/homa/src/lib.rs",
     [RUSTFMT, "--check", "--edition", "2021", P.root / "crates" / "homa" / "src" / "lib.rs"]),
    ("rustfmt --check --edition 2024 crates/aosabi-devtool/src/main.rs",
     [RUSTFMT, "--check", "--edition", "2024", P.root / "crates" / "aosabi-devtool" / "src" / "main.rs"]),
]
TARGETS = run([RUSTUP, "target", "list", "--installed"], cwd=P.root, timeout_s=60).out.split()
GATE_FAILURES = []
for label, argv in GATES:
    gate = run(argv, cwd=P.root, timeout_s=900, check=False)
    print("gate %-66s exit %d, %.1f s" % (label, gate.returncode, gate.duration_s))
    if gate.returncode != 0:
        GATE_FAILURES.append(label)
        print((gate.err or gate.out).strip()[-2000:])

reported_cargo = run([CARGO, "--version"], cwd=P.root, timeout_s=timeout_s).out.strip()
smoke = json.loads(run([DEVTOOL, "homa-sim", "--smoke"], cwd=P.root, timeout_s=timeout_s).out)
print("repository root:", P.rel(P.root))
print("devtool        :", P.rel(DEVTOOL))
print("cargo          :", reported_cargo)
print("targets        :", ", ".join(TARGETS))
print("seam           :", json.dumps(smoke))
print("seed           :", seed)
assert not GATE_FAILURES, GATE_FAILURES
assert rust_version in reported_cargo, reported_cargo
DROP_RATES = [int(rate) for rate in drop_rates.split(",")]
assert half in ("engine", "socket"), half
assert smoke == {"subcommand": "homa-sim", "implemented": True, "lands_at": "H0"}, smoke

PANICS = []
NAMED = {"LinkDown", "PeerTimeout", "TooLong", "Aborted", "RpcUnknown"}
TOTALS = {"rpcs": 0, "ended": 0, "named_failures": 0, "ended_by_the_client": 0, "unnamed": 0, "duplicates": 0,
          "payload_differences": 0, "restarts_after_answer": 0, "unattributed_restarts": 0, "runs_not_done": 0}


def sim(requests, name):
    """Send scenarios to `homa-sim` in one process and return its answers."""
    path = OUT / ("%s.ndjson" % name)
    path.write_text("".join(json.dumps(r) + "\n" for r in requests), encoding="utf-8")
    result = run(
        [DEVTOOL, "homa-sim", "--requests", path], cwd=P.root, timeout_s=timeout_s, check=False
    )
    answers = [json.loads(line) for line in result.out.splitlines()]
    if result.returncode != 0 or len(answers) != len(requests):
        PANICS.append("%s: exit %d, %d answers to %d requests, %s" % (
            name, result.returncode, len(answers), len(requests), result.err.strip()[-300:]))
    bad = [a for a in answers if "fatal" in a or "config_error" in a]
    assert not bad, "the seam refused scenarios of %s: %s" % (name, bad[:2])
    return answers


def fill(spec, length):
    """The bytes of a message, as the devtool's scenario defines a fill."""
    if "hex" in spec:
        pattern = bytes.fromhex(spec["hex"])
        return (pattern * (length // max(len(pattern), 1) + 1))[:length]
    out = bytearray()
    block = 0
    while len(out) < length:
        out += hashlib.sha256(spec["seed"].to_bytes(8, "little") + block.to_bytes(8, "little")).digest()
        block += 1
    return bytes(out[:length])


def digest(spec, length):
    return hashlib.sha256(fill(spec, length)).hexdigest()


def tally(scenario, answer):
    """Count what every RPC of a run did, against the bytes it was asked for."""
    TOTALS["rpcs"] += len(scenario["rpcs"])
    if answer["stopped"] != "done":
        TOTALS["runs_not_done"] += 1
    # A restart no packet of the run names escapes the count of restarts after
    # a byte of the answer, so the verdict holds it to 0 as well.
    TOTALS["unattributed_restarts"] += sum(
        answer["links"][side]["unattributed_restarts"] for side in ("A", "B"))
    for index, (spec, got) in enumerate(zip(scenario["rpcs"], answer["rpcs"])):
        if got["outcome"] == "pending":
            continue
        TOTALS["ended"] += 1
        if got["outcome"] == "ok":
            request = spec.get("fill", {"seed": 2 * index})
            response = spec.get("response_fill", {"seed": 2 * index + 1})
            if got["request_sha"] != digest(request, spec["len"]) or \
                    got["response_sha"] != digest(response, spec["response_len"]):
                TOTALS["payload_differences"] += 1
        elif got["outcome"] == "aborted":
            TOTALS["ended_by_the_client"] += 1
        elif got["why"] in NAMED:
            TOTALS["named_failures"] += 1
        else:
            TOTALS["unnamed"] += 1
        if got["server_requests"] > 1 or got["client_responses"] > 1:
            TOTALS["duplicates"] += 1
        TOTALS["restarts_after_answer"] += got["restarts_after_answer"]


PROFILES = {
    "serial_line": ({"name": "serial_line", "baud": 115200},
                    {"bytes_per_s": 11_520, "packets_per_event": 1, "event_ns": 0, "latency_ns": 100_000}),
    "bluetooth_gatt": ({"name": "bluetooth_gatt", "att_mtu": 185, "interval_ns": 30_000_000},
                       {"bytes_per_s": 12_133, "packets_per_event": 2, "event_ns": 30_000_000, "latency_ns": 1_000_000}),
    "usb_serial": ({"name": "usb_serial"},
                   {"bytes_per_s": 1_000_000, "packets_per_event": 2, "event_ns": 0, "latency_ns": 1_000_000}),
    "unix_socket": ({"name": "unix_socket", "max_message": 65536},
                    {"bytes_per_s": 0, "packets_per_event": 2, "event_ns": 0, "latency_ns": 10_000}),
}
CONFIG = {"client_slots": 8, "server_slots": 8, "page_len": 4096, "pages": 256, "max_gaps": 2}


def scenario(ident, profile, rpcs, **extra):
    spec, model = PROFILES[profile]
    body = {"id": ident, "op": "run", "link": "lossy", "profile": spec, "model": model,
            "config": CONFIG, "window": 4, "seed": seed + ident, "rpcs": rpcs,
            "max_time_ns": 7_200 * 1_000_000_000}
    body.update(extra)
    return body

## The loss sweep

Each of the `seeds` seeds runs at each drop rate, 0, 0.1, 1, 5 and 20 percent of the packets, with 1
percent of the packets duplicated, 1 percent damaged, which flips one bit of the packet's delimited
AoFrame and reads it back, and every GRANT of 10 percent of the messages lost, over the Serial Line,
GATT, USB serial and Unix Socket profiles in turn. Every tenth seed also loses the first LIMITS each
end sends. Each run carries `rpcs_per_run` RPCs of 1 to 6,000 bytes each way, beyond the unscheduled
bytes of every profile but the Unix Socket, so most messages need GRANTs.

A lost GRANT case is a message that needs a grant and whose receiver sent GRANTs of which not one
reached its sender. A sender sends a packet whole once its first byte is granted, which rounds the
unscheduled bytes up to whole packets, so a message needs a grant where it is longer than the
payloads of the packets its unscheduled bytes open. The sender of such a message then holds the
bytes it was granted and no more, and only a RESEND of its receiver, which raises the granted bytes
to the range it names on a point to point profile, moves it on. A case recovers when its RPC
completes after its receiver sent a RESEND for the message.

In [ ]:
if half == "engine":
    g = random.Random(seed)
    names = list(PROFILES)
    sweep = []
    for s in range(seeds):
        profile = names[s % len(names)]
        for rate in DROP_RATES:
            rpcs = [{"len": g.randint(1, 6000), "response_len": g.randint(1, 6000)} for _ in range(rpcs_per_run)]
            sweep.append(scenario(
                len(sweep), profile, rpcs,
                faults={"drop_per_million": rate, "duplicate_per_million": 10_000,
                        "corrupt_per_million": 10_000, "drop_grants_per_million": 100_000},
                drop_first_limits=(s % 10 == 0),
            ))
    answers = sim(sweep, "sweep")
    lost_grant_cases = 0
    recovered = 0
    by_rate = {}
    limits_dropped = 0
    injected = {}
    def unscheduled_reach(profile):
        """The bytes the packets that the unscheduled bytes open carry, whole."""
        per_packet = profile["max_packet"] - 14
        return -(-profile["unsched_bytes"] // per_packet) * per_packet


    for body, answer in zip(sweep, answers):
        tally(body, answer)
        reach = unscheduled_reach(answer["profile"])
        rate = body["faults"]["drop_per_million"]
        row = by_rate.setdefault(rate, {"rpcs": 0, "ok": 0, "failed": {}, "simulated_s": 0.0, "resends": 0})
        row["simulated_s"] += answer["end_ns"] / 1e9
        row["resends"] += answer["stats"]["A"]["resends"] + answer["stats"]["B"]["resends"]
        for key, value in answer["injected"].items():
            injected[key] = injected.get(key, 0) + value
        limits_dropped += answer["links"]["A"]["dropped"] + answer["links"]["B"]["dropped"]
        for spec, got in zip(body["rpcs"], answer["rpcs"]):
            row["rpcs"] += 1
            if got["outcome"] == "ok":
                row["ok"] += 1
            else:
                row["failed"][got["why"]] = row["failed"].get(got["why"], 0) + 1
            request_sent, request_taken, response_sent, response_taken = got["grants"]
            for length, sent, taken, resends in (
                (spec["len"], request_sent, request_taken, got["resends"][1]),
                (spec["response_len"], response_sent, response_taken, got["resends"][0]),
            ):
                if length > reach and sent > 0 and taken == 0:
                    lost_grant_cases += 1
                    recovered += got["outcome"] == "ok" and resends > 0
    print("%-10s %-6s %-6s %-22s %-14s %s" % ("drop ppm", "RPCs", "ok", "failed", "simulated s", "RESENDs"))
    for rate, row in sorted(by_rate.items()):
        print("%-10d %-6d %-6d %-22s %-14.1f %d" % (rate, row["rpcs"], row["ok"], row["failed"], row["simulated_s"], row["resends"]))
    print()
    print("faults injected over the sweep:", injected)
    print("first LIMITS dropped by the scenario:", limits_dropped)
    print("lost GRANT cases: %d, recovered through RESEND: %d" % (lost_grant_cases, recovered))

## Lost requests and `report_unknown`

A request whose every packet is lost until it has left whole once reaches no server. The client asks
for its answer after its RESEND time, the server knows no such RPC and answers `RPC_UNKNOWN`, and
the client sends the request again from offset 0 where the request did not ask for the report, as
`HomaModule` does, and ends the RPC with `Failure::RpcUnknown` where it did, counted in
`Stats::unknown_reported`. Each value runs `lost_requests` such requests beside as many that arrive,
on the Serial Line profile over a perfect link.

In [ ]:
if half == "engine":
    unknown = {}
    for report in (False, True):
        bodies = []
        for n in range(lost_requests // 10):
            rpcs = []
            for k in range(20):
                rpcs.append({"len": g.randint(1, 900), "response_len": g.randint(1, 900), "lose": k % 2 == 0})
            body = scenario(1000 + n, "serial_line", rpcs, report_unknown=report)
            body["link"] = "loopback"
            bodies.append(body)
        answers = sim(bodies, "unknown-%s" % str(report).lower())
        row = {"lost": 0, "restarted_and_ok": 0, "failed_rpc_unknown": 0, "restarts": 0,
               "unknown_reported": 0, "others_ok": 0, "others": 0}
        for body, answer in zip(bodies, answers):
            tally(body, answer)
            row["unknown_reported"] += answer["stats"]["A"]["unknown_reported"]
            for spec, got in zip(body["rpcs"], answer["rpcs"]):
                if spec["lose"]:
                    row["lost"] += 1
                    row["restarts"] += got["restarts"]
                    row["restarted_and_ok"] += got["restarts"] >= 1 and got["outcome"] == "ok"
                    row["failed_rpc_unknown"] += got["outcome"] == "failed" and got["why"] == "RpcUnknown" and got["restarts"] == 0
                else:
                    row["others"] += 1
                    row["others_ok"] += got["outcome"] == "ok"
        unknown[report] = row
        print("report_unknown %-5s %s" % (report, row))

## The link down in the middle of a message

The link goes down at a seeded time while a message of 16 KiB moves on the GATT and the Serial Line
profile, and the scenario calls `link_down` on both ends at that time, since a link reports nothing
to the engine. Every RPC that has not completed fails with `Failure::LinkDown` at that time and the
peer is reported gone once.

In [ ]:
if half == "engine":
    down_rows = []
    bodies = []
    for n in range(20):
        profile = ("bluetooth_gatt", "serial_line")[n % 2]
        rpcs = [{"len": g.randint(1, 500), "response_len": 16384} for _ in range(4)]
        at = g.randint(300, 2500) * 1_000_000
        body = scenario(2000 + n, profile, rpcs, faults={"link_down_at_ns": at})
        bodies.append(body)
    answers = sim(bodies, "link-down")
    down_ok = 0
    for body, answer in zip(bodies, answers):
        tally(body, answer)
        at = body["faults"]["link_down_at_ns"]
        outcomes = [(got["outcome"], got["why"]) for got in answer["rpcs"]]
        failed_at_down = all(
            got["outcome"] == "ok" or (got["why"] == "LinkDown" and got["ended_ns"] == at)
            for got in answer["rpcs"]
        )
        gone = answer["links"]["A"]["gone_link_down"]
        down_ok += failed_at_down and gone == 1 and answer["stopped"] == "done"
        down_rows.append((body["profile"]["name"], at // 1_000_000, outcomes, gone))
    for profile, at, outcomes, gone in down_rows[:6]:
        print("%-16s down at %5d ms: %s, PeerGone %d" % (profile, at, outcomes, gone))
    print("runs whose open RPCs failed LinkDown at the time the link went down: %d of %d" % (down_ok, len(bodies)))

## The message that holds the grant ends first

A downlink runs its grant scheduler again when the incoming message that holds its grant ends before
it is whole, so the message behind it takes the grant and never waits for ever. Each case ends the
first message on the Serial Line profile while a second one waits for its grant: the client ends a
response of 3,000 bytes at 235 ms through `Endpoint::abort`, `Endpoint::release`, `Rpcs::abort` and
`Rpcs::release`, and ends its own request of 3,000 bytes at 124 ms through `Endpoint::abort`, which
the server drops on `RPC_UNKNOWN`. The second message, a response or a request of 4,000 bytes, must
complete.

In [ ]:
if half == "engine":
    cases = []
    for by in ("abort", "release", "rpcs_abort", "rpcs_release"):
        cases.append((by, "response", [{"len": 10, "response_len": 3000, "end_at_ns": 235_000_000, "end_by": by},
                                       {"len": 10, "response_len": 4000}]))
    cases.append(("abort", "request", [{"len": 3000, "response_len": 10, "end_at_ns": 124_000_000, "end_by": "abort"},
                                       {"len": 4000, "response_len": 10}]))
    bodies = []
    for n, (_, _, rpcs) in enumerate(cases):
        body = scenario(3000 + n, "serial_line", rpcs)
        body["link"] = "loopback"
        bodies.append(body)
    answers = sim(bodies, "grant-holder")
    second_done = 0
    for (by, which, _), body, answer in zip(cases, bodies, answers):
        tally(body, answer)
        first, second = answer["rpcs"]
        done = second["outcome"] == "ok"
        second_done += done
        print("%-13s the %-8s of the first RPC ended: first %s %s, second %s at %s ms" % (
            by, which, first["outcome"], first["why"], second["outcome"],
            None if second["ended_ns"] is None else round(second["ended_ns"] / 1e6, 1)))
    print("second messages completed: %d of %d" % (second_done, len(cases)))

## A silent server and a patient request

A server that takes and sends nothing leaves the client's RESENDs unanswered, and the client ends
the RPC with `Failure::PeerTimeout` once `timeout_resends` of them went unanswered and the peer was
silent for `rpc_timeout`, which on the Serial Line profile is 5 RESENDs 550 ms apart. Its bound is
the first RESEND, the RESENDs after it and `rpc_timeout` from the last packet the client heard. A
patient request that the server holds for 30 s is asked after `patient_after`, 5 s, answered BUSY
and never declared dead.

In [ ]:
if half == "engine":
    silent_body = scenario(4000, "serial_line", [{"len": 3000, "response_len": 10}], silent_b_from_ns=0)
    silent_body["link"] = "loopback"
    patient_body = scenario(4001, "serial_line", [{"len": 10, "response_len": 30, "patient": True, "hold_ns": 30_000_000_000}])
    patient_body["link"] = "loopback"
    silent, patient = sim([silent_body, patient_body], "silent-patient")
    tally(silent_body, silent)
    tally(patient_body, patient)
    timers = silent["profile"]["timers"]
    bound = timers["resend_after_ns"] + timers["timeout_resends"] * timers["resend_interval_ns"] + timers["rpc_timeout_ns"]
    silent_rpc = silent["rpcs"][0]
    silent_ok = silent_rpc["why"] == "PeerTimeout" and silent_rpc["ended_ns"] <= bound
    print("silent server: %s %s at %.0f ms after %d RESENDs, bound %.0f ms" % (
        silent_rpc["outcome"], silent_rpc["why"], silent_rpc["ended_ns"] / 1e6,
        silent["stats"]["A"]["resends"], bound / 1e6))
    patient_rpc = patient["rpcs"][0]
    patient_ok = patient_rpc["outcome"] == "ok" and patient["stats"]["A"]["peer_deaths"] == 0
    print("patient request held 30 s: %s at %.0f ms, RESENDs %d, deaths %d" % (
        patient_rpc["outcome"], patient_rpc["ended_ns"] / 1e6, patient["stats"]["A"]["resends"],
        patient["stats"]["A"]["peer_deaths"]))

## The socket half: loss on a real socket

`aosabi-devtool homa-unix serve` runs the engine of the crate on a real Unix Socket, one server
endpoint for the connection, and answers every request with its own bytes. A client of the Python
engine `tests/harness/homa/client.py` sends `socket_rpcs` requests of 1 to 16 KiB through a hook
that drops each packet with the rate of the run and duplicates `socket_duplicate_ppm` per million,
in both directions, at 1 and 5 percent loss, on both forms, with the seed of the parameters cell. A
socket loses nothing of itself, so the hook is every loss of the run. Both ends run the timers of
the Unix Socket profile shortened to a RESEND after 20 ms, since the 1,000 ms of the profile would
make a run of losses last minutes, and a peer is dead after 10 unanswered questions and 200 ms of
silence. The cell counts every RPC that completed, with its bytes against the request, every one
that failed with a named failure, and every request the server delivered twice.

A pty pair of `tests/harness/homa/pty.py` carries `pty_rpcs` RPCs of the Serial Line profile in
delimited AoFrame between two endpoints of the crate in one process, through `homa-sim` and its link
`pty`, whose ends open the two slaves of the pair.


In [ ]:
SOCKET = {}
if half == "socket":
    import subprocess
    import threading
    import time
    from harness.homa import client as hc
    from harness.homa import pty as hpty

    SOCKS = OUT / "sock"
    SOCKS.mkdir(parents=True, exist_ok=True)
    UNIX_SPEC = {"name": "unix_socket", "max_message": 65536}
    UNIX_LIMITS = {"max_packet": 16384, "max_message": 65536, "unsched_bytes": 65536, "max_rpcs": 16}
    FAST = {"resend_after_ns": 20_000_000, "resend_interval_ns": 20_000_000, "need_ack_after_ns": 40_000_000,
            "rpc_timeout_ns": 200_000_000, "timeout_resends": 10}
    unix_smoke = json.loads(run([DEVTOOL, "homa-unix", "--smoke"], cwd=P.root, timeout_s=timeout_s).out)
    print("seam homa-unix :", json.dumps(unix_smoke))
    assert unix_smoke == {"subcommand": "homa-unix", "implemented": True, "lands_at": "H2"}, unix_smoke

    def fast_profile():
        return hc.unix_socket(65536).override(resend_after=0.02, resend_interval=0.02, need_ack_after=0.04,
                                              rpc_timeout=0.2, timeout_resends=10)

    def serve(request, name):
        """Start `homa-unix serve` in the background and wait for its socket."""
        sock = Path(request["path"])
        if sock.exists():
            sock.unlink()
        path = OUT / ("%s.ndjson" % name)
        path.write_text(json.dumps(dict(request, id=0, op="serve")) + "\n", encoding="utf-8")
        proc = subprocess.Popen([str(DEVTOOL), "homa-unix", "--requests", str(path)], cwd=P.root,
                                stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        hc.wait_for(sock)
        return proc

    def served(proc, name):
        """Wait for a server of `serve` and return its one connection."""
        out, err = proc.communicate(timeout=timeout_s)
        answers = [json.loads(line) for line in out.splitlines()]
        if proc.returncode != 0 or len(answers) != 1 or "fatal" in answers[0] or not answers[0]["connections"]:
            PANICS.append("%s: exit %d, %s, %s" % (name, proc.returncode, out.strip()[-300:], err.strip()[-300:]))
            return {}
        return answers[0]["connections"][0]

    def py_client(form, path, hook=None):
        profile = fast_profile()
        transport = hc.open_transport(form, path, profile.max_packet, hook)
        return hc.Engine(transport, profile, hc.limits(profile, 65536), UNIX_LIMITS)

    gl = random.Random(seed ^ 0x2210)
    for form in ("seqpacket", "stream"):
        for rate in [int(r) for r in socket_rates.split(",")]:
            name = "loss-%s-%d" % (form, rate)
            messages = []
            for n in range(socket_rpcs):
                length = gl.randint(1, 16384)
                if n % 5 == 4:
                    text = "日本語のテキスト、漢字とひらがなとカタカナ。".encode()
                    messages.append((text * (length // len(text) + 1))[:length])
                else:
                    messages.append(gl.randbytes(length))
            path = str(SOCKS / ("%s.sock" % name))
            proc = serve({"form": form, "path": path, "profile": UNIX_SPEC, "overrides": FAST, "responder": "echo",
                          "connections": 1}, name)
            hook = hc.Loss(seed + rate + (0 if form == "seqpacket" else 1), drop_ppm=rate,
                           duplicate_ppm=socket_duplicate_ppm)
            engine = py_client(form, path, hook)
            started = time.monotonic()
            rpcs = hc.run_requests(engine, messages, window=4, timeout=timeout_s / 4)
            elapsed = time.monotonic() - started
            engine.t.sock.close()
            conn = served(proc, name)
            named = {"LinkDown", "PeerTimeout", "TooLong", "Aborted", "RpcUnknown"}
            row = {
                "rpcs": len(rpcs),
                "ok": sum(r.state == "ok" for r in rpcs),
                "named_failures": sum(r.state == "failed" and r.why in named for r in rpcs),
                "unnamed": sum(r.state not in ("ok", "failed") or (r.state == "failed" and r.why not in named) for r in rpcs),
                "payload_differences": sum(1 for r in rpcs if r.state == "ok" and r.response != messages[r.index]),
                "server_duplicates": conn.get("duplicates"),
                "server_requests": conn.get("requests"),
                "client_restarts": engine.stats["restarts"],
                "client_resends": engine.stats["resends"],
                "server_resends": conn.get("stats", {}).get("resends"),
                "hook": dict(hook.counts),
                "seconds": round(elapsed, 2),
            }
            SOCKET[(form, rate)] = row
            print("%-9s loss %5.1f %%: %s" % (form, rate / 1e4, row))

    pair = hpty.PtyPair()
    try:
        gp = random.Random(seed ^ 0x2220)
        pty_specs = [{"len": gp.randint(1, 2048), "response_len": gp.randint(1, 2048)} for _ in range(pty_rpcs)]
        pty_answer = sim([{"id": 0, "op": "run", "link": "pty", "pty_a": pair.path_a, "pty_b": pair.path_b,
                           "profile": {"name": "serial_line", "baud": 115200}, "config": CONFIG, "window": 4,
                           "seed": seed, "max_time_ns": 600 * 1_000_000_000, "rpcs": pty_specs}], "pty")[0]
    finally:
        pair.close()
    PTY = {"rpcs": len(pty_answer["rpcs"]), "ok": sum(r["outcome"] == "ok" for r in pty_answer["rpcs"]),
           "payload_differences": sum(1 for n, (spec, r) in enumerate(zip(pty_specs, pty_answer["rpcs"]))
                                      if r["outcome"] == "ok" and (r["request_sha"] != digest({"seed": 2 * n}, spec["len"])
                                                                   or r["response_sha"] != digest({"seed": 2 * n + 1}, spec["response_len"]))),
           "duplicates": sum(1 for r in pty_answer["rpcs"] if r["server_requests"] > 1 or r["client_responses"] > 1),
           "stopped": pty_answer["stopped"], "relayed": pair.moved}
    print("pty pair, Serial Line profile:", PTY)


## The socket half: the server rebuilt and the silent client

**The server rebuilt.** `homa-unix serve` with `rebuild_after` 1 drops the engine of the connection
once the first request has arrived whole, before it answers, and builds a new one over the same open
socket with the first identifier `rebuild_first_id`, so the request is in flight at the Homa layer
and the new engine knows nothing of it. The client of `client.py` asks for its answer with a RESEND
after 20 ms, and the new engine answers `RPC_UNKNOWN`. Under `report_unknown` false the client sends
the request again from offset 0, and the new engine delivers it once and answers it once. Under true
the client ends the RPC with the failure `RpcUnknown` and restarts nothing, so the request ran at
most once at the Homa layer, which is the rule of `docs/graphtp.md` section 2.7. Both values run on
both forms.

**The silent client.** A client sends the first packet of a request of 40,000 bytes, the first of
three packets, and falls silent: it reads nothing and sends nothing. The server asks for the rest
with a RESEND every 20 ms, and after 10 unanswered questions and 200 ms of silence declares the peer
dead, drops its server RPC and reports `Event::PeerGone` with `Gone::Timeout`. The client then
closes its socket, and the link down the server reports after it is printed beside.


In [ ]:
if half == "socket":
    REBUILD = {}
    for form in ("seqpacket", "stream"):
        for report in (False, True):
            name = "rebuild-%s-%s" % (form, str(report).lower())
            path = str(SOCKS / ("%s.sock" % name))
            proc = serve({"form": form, "path": path, "profile": UNIX_SPEC, "overrides": FAST, "responder": "echo",
                          "connections": 1, "rebuild_after": 1, "rebuild_first_id": 0x21000002}, name)
            engine = py_client(form, path)
            message = "再構築の間に飛ぶ要求, a request in flight across the rebuild".encode()
            rpc = engine.request(message, report_unknown=report)
            deadline = time.monotonic() + 30
            while not engine.done([rpc]) and time.monotonic() < deadline:
                engine.poll(0.002)
            engine.t.sock.close()
            conn = served(proc, name)
            REBUILD[(form, report)] = {
                "state": rpc.state, "why": rpc.why, "restarts": rpc.restarts,
                "unknown_reported": rpc.unknown_reported, "response_ok": rpc.response == message,
                "server_requests": conn.get("requests"), "server_answered": conn.get("answered"),
                "rebuilds": conn.get("rebuilds"), "duplicates": conn.get("duplicates"),
            }
            print("%-9s report_unknown %-5s %s" % (form, report, REBUILD[(form, report)]))

    path = str(SOCKS / "silent.sock")
    proc = serve({"form": "seqpacket", "path": path, "profile": UNIX_SPEC, "overrides": FAST, "responder": "echo",
                  "connections": 1}, "silent")
    engine = py_client("seqpacket", path)
    engine.data_budget = 1
    partial = engine.request(random.Random(seed ^ 0x2230).randbytes(40_000))
    deadline = time.monotonic() + 10
    while engine.stats["data_sent"] < 1 and time.monotonic() < deadline:
        engine.poll(0.002)
    engine.silent = True
    time.sleep(1.5)
    engine.t.sock.close()
    conn = served(proc, "silent")
    SILENT = {"gone": conn.get("gone"), "dropped": conn.get("dropped"), "gone_ms": conn.get("gone_ms"),
              "peer_deaths": conn.get("stats", {}).get("peer_deaths"), "server_resends": conn.get("stats", {}).get("resends"),
              "first_packet_sent": engine.stats["data_sent"]}
    print("silent client:", SILENT)


## Verdict

In [ ]:
print("panics or crashes of the seam:", len(PANICS))
for panic in PANICS:
    print("  ", panic)
if half == "engine":
    print("RPCs:", TOTALS)
    false_row, true_row = unknown[False], unknown[True]
    print("lost requests under report_unknown false: %d restarted and completed of %d" % (false_row["restarted_and_ok"], false_row["lost"]))
    print("lost requests under report_unknown true: %d failed RpcUnknown with 0 restarts of %d, unknown_reported %d" % (
        true_row["failed_rpc_unknown"], true_row["lost"], true_row["unknown_reported"]))
    print("lost GRANT cases recovered through RESEND: %d of %d" % (recovered, lost_grant_cases))
    assert not PANICS, PANICS
    assert TOTALS["ended"] == TOTALS["rpcs"] and TOTALS["runs_not_done"] == 0, TOTALS
    assert TOTALS["unnamed"] == 0 and TOTALS["duplicates"] == 0 and TOTALS["payload_differences"] == 0, TOTALS
    assert TOTALS["restarts_after_answer"] == 0 and TOTALS["unattributed_restarts"] == 0, TOTALS
    assert lost_grant_cases >= 100 and recovered == lost_grant_cases, (recovered, lost_grant_cases)
    assert false_row["lost"] == lost_requests and false_row["restarted_and_ok"] == lost_requests, false_row
    assert true_row["lost"] == lost_requests and true_row["failed_rpc_unknown"] == lost_requests, true_row
    assert true_row["unknown_reported"] == lost_requests and true_row["restarts"] == 0, true_row
    assert false_row["others_ok"] == false_row["others"] and true_row["others_ok"] == true_row["others"]
    assert down_ok == 20 and second_done == len(cases) and silent_ok and patient_ok
    print()
    print("VERDICT: %d RPCs, every one completed or failed with a named failure within its bound,"
          " 0 duplicate deliveries, %d of %d lost GRANT cases recovered through RESEND, %d of %d lost"
          " requests restarted and completed under report_unknown false and %d of %d failed"
          " RpcUnknown with 0 restarts under true, 0 restarts after a byte of the answer, and the"
          " socket half not run" % (
              TOTALS["rpcs"], recovered, lost_grant_cases, false_row["restarted_and_ok"], lost_requests,
              true_row["failed_rpc_unknown"], lost_requests))
if half == "socket":
    assert not PANICS, PANICS
    for key, row in SOCKET.items():
        print("%s at %d ppm: %s" % (key[0], key[1], row))
    for row in SOCKET.values():
        assert row["unnamed"] == 0 and row["ok"] + row["named_failures"] == row["rpcs"] == socket_rpcs, row
        assert row["payload_differences"] == 0 and row["server_duplicates"] == 0, row
    assert PTY["ok"] == pty_rpcs and PTY["payload_differences"] == 0 and PTY["duplicates"] == 0, PTY
    for form in ("seqpacket", "stream"):
        off, on = REBUILD[(form, False)], REBUILD[(form, True)]
        assert off["state"] == "ok" and off["response_ok"] and off["restarts"] == 1, off
        assert off["server_requests"] == [1, 1] and off["server_answered"] == [0, 1] and off["rebuilds"] == 1, off
        assert on["state"] == "failed" and on["why"] == "RpcUnknown" and on["restarts"] == 0, on
        assert on["unknown_reported"] == 1 and on["server_answered"] == [0, 0], on
    assert SILENT["gone"] and SILENT["gone"][0] == "Timeout" and SILENT["dropped"] >= 1, SILENT
    assert SILENT["peer_deaths"] == 1 and SILENT["first_packet_sent"] == 1, SILENT
    rpcs = sum(row["rpcs"] for row in SOCKET.values())
    ok = sum(row["ok"] for row in SOCKET.values())
    print()
    print("VERDICT: %d RPCs over real Unix Sockets at 1 and 5 percent loss on both forms, %d completed and"
          " the rest failed with a named failure, 0 payload differences, 0 duplicate deliveries, %d of %d"
          " RPCs over a pty pair, 1 of 1 request in flight across the server rebuild restarted and"
          " answered once under report_unknown false and failed RpcUnknown with 0 restarts under true on"
          " each form, 1 of 1 silent client declared dead with its server RPC dropped and PeerGone"
          " reported, and the engine half not run" % (rpcs, ok, PTY["ok"], pty_rpcs))
